# Chapter 1: Microservices Architecture Fundamentals

This notebook demonstrates the foundational architectural patterns that
make microservices resilient and agent-ready.

**Concepts covered:**
- Circuit Breaker (3-state machine)
- Bulkhead isolation
- Service health checking
- Exponential back-off
- Sidecar / Strangler Fig patterns (conceptual)

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))
from core.patterns.circuit_breaker import (
    CircuitBreaker, CircuitBreakerError, CircuitState,
    AgentCircuitBreaker,
)
print('Core package loaded successfully.')

## 1.1 The Circuit Breaker Pattern

The circuit breaker prevents cascading failures by tracking error rates and
**temporarily rejecting calls** to a degraded service.

State machine: `CLOSED` → `OPEN` (on threshold) → `HALF_OPEN` (after timeout) → `CLOSED` (on recovery).

In [ ]:
import asyncio

cb = CircuitBreaker(failure_threshold=3, recovery_timeout_s=1.0)
print(f'Initial state: {cb.state.name}')  # CLOSED

# Simulate 3 downstream failures
async def simulate_failures():
    for i in range(3):
        try:
            await cb.call(lambda: (_ for _ in ()).throw(RuntimeError('service unavailable')))
        except RuntimeError as e:
            print(f'  Failure {i+1}: {e}')
    print(f'State after failures: {cb.state.name}')  # OPEN

    # Try one more call - should be immediately rejected
    try:
        await cb.call(lambda: None)
    except CircuitBreakerError as e:
        print(f'  Rejected call: {e}')

asyncio.run(simulate_failures())

## 1.2 Half-Open Recovery

After `recovery_timeout_s`, the breaker enters `HALF_OPEN` — it allows **one probe call**.
If the probe succeeds, the breaker resets to `CLOSED`. If it fails, it returns to `OPEN`.

In [ ]:
import asyncio, time

async def demo_recovery():
    cb = CircuitBreaker(failure_threshold=2, recovery_timeout_s=0.05)
    # Trip the breaker
    for _ in range(2):
        try: await cb.call(lambda: (_ for _ in ()).throw(RuntimeError('error')))
        except RuntimeError: pass
    print(f'After trips: {cb.state.name}')

    # Wait for recovery timeout
    await asyncio.sleep(0.1)

    # One successful probe
    async def healthy_service():
        return 'healthy'
    result = await cb.call(healthy_service)
    print(f'Probe result: {result}')
    print(f'After recovery: {cb.state.name}')  # CLOSED again

asyncio.run(demo_recovery())

## 1.3 Bulkhead with Semaphore

Bulkhead isolates tenants so one slow consumer can't starve others.
In Python, `asyncio.Semaphore` provides this guarantee.

In [ ]:
import asyncio

class Bulkhead:
    def __init__(self, max_concurrent: int):
        self._sem = asyncio.Semaphore(max_concurrent)
        self.rejected = 0

    async def execute(self, fn):
        if self._sem._value == 0:
            self.rejected += 1
            raise RuntimeError('Bulkhead full')
        async with self._sem:
            return await fn()

async def demo_bulkhead():
    bh = Bulkhead(max_concurrent=2)
    results = []
    async def slow(): await asyncio.sleep(0.05); return 'ok'
    # Launch 5 concurrent calls - only 2 can run at once
    tasks = [asyncio.create_task(bh.execute(slow)) for _ in range(5)]
    for t in tasks:
        try: results.append(await t)
        except RuntimeError as e: results.append(f'REJECTED: {e}')
    print(f'Results: {results}')
    print(f'Rejected: {bh.rejected}')

asyncio.run(demo_bulkhead())

## 1.4 AI-Specific Circuit Breaker

The `AgentCircuitBreaker` adds three LLM-specific failure modes:
- **Latency SLO**: trips if p95 latency exceeds threshold
- **Confidence gate**: rejects calls below minimum confidence score
- **Token budget**: cuts off spending above a session limit

In [ ]:
import asyncio

async def demo_agent_circuit_breaker():
    acb = AgentCircuitBreaker(
        latency_slo_ms=5_000,
        confidence_threshold=0.7,
        token_budget=1_000,
    )
    print(f'Initial state: {acb.state.name}')

    async def mock_llm_call():
        return {'answer': 'Microservices isolate bounded contexts.', 'tokens_used': 42}

    # Successful call with confidence above threshold
    result = await acb.call(mock_llm_call, expected_confidence=0.9, token_cost=42)
    print(f'Tokens consumed: {acb.tokens_consumed}')
    print(f'State: {acb.state.name}')

    # Budget exhaustion
    try:
        await acb.call(mock_llm_call, expected_confidence=0.9, token_cost=10_000)
    except CircuitBreakerError as e:
        print(f'Budget exhausted: {e}')

asyncio.run(demo_agent_circuit_breaker())

## 1.5 Azure Deployment Note

In production on **Azure Container Apps** (the natural home for agentic microservices):
- Each agent runs as an independent Container App
- Dapr is enabled for service-to-service calls and pub/sub
- The `CircuitBreaker` above is implemented *in-process*; Dapr's built-in
  circuit-breaking handles the transport layer
- See `infra/modules/container-apps.bicep` to provision the environment.

```bash
# Deploy Infrastructure
az deployment group create \
    --resource-group rg-agentic-showcase \
    --template-file infra/main.bicep \
    --parameters infra/parameters/dev.bicepparam
```